# Part 1: Regression Task (California Housing)

## Task 1: Load and Split Dataset

In [19]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Setting the columns 
cols = ["longitude", "latitude", "housingMedianAge", "totalRooms", "totalBedrooms", "population", "households", "medianIncome", "medianHouseValue"]

# Loading the dataset by downloading from "https://s3-eu-west-1.amazonaws.com/pfigshare-u-files/5976036/cal_housing.tgz"
df = pd.read_csv("cal_housing.data", header=None, names=cols)

# Spliting the data into features and label
X = df.drop("medianHouseValue", axis = 1)
y = df["medianHouseValue"]
# Split into training (80%) and test (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\nTraining set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Features: {X_train.shape[1]}")


FileNotFoundError: [Errno 2] No such file or directory: 'cal_housing.data'

## Task 2, Step 1: Baseline Model (No Regularization)

In [9]:

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Build Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)

# Observe coefficients
print("Coefficients:")
for i in range(min(5, len(model.coef_))):  # Show first 5
    print(f"  Feature {i}: {model.coef_[i]:.6f}")
print(f"  ... and {len(model.coef_) - 5} more")
print(f"Intercept: {model.intercept_:.6f}")

# Compute MSE
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

mse_train = mean_squared_error(y_train, y_train_pred)
mse_test = mean_squared_error(y_test, y_test_pred)

print(f"\nTraining MSE: {mse_train:.6f}")
print(f"Test MSE: {mse_test:.6f}")


NameError: name 'X_train' is not defined

## Task 2, Step 2: Hyperparameter Tuning

In [10]:

from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import GridSearchCV

# Define alphas
alphas = [0.001, 0.01, 0.1, 1, 10, 100, 1000]

# Ridge Regression tuning
ridge = Ridge()
ridge_grid = GridSearchCV(ridge, {'alpha': alphas}, cv=5, scoring='neg_mean_squared_error')
ridge_grid.fit(X_train, y_train)

print(f"Ridge - Best alpha: {ridge_grid.best_params_['alpha']}")

# Lasso Regression tuning
lasso = Lasso(max_iter=10000)
lasso_grid = GridSearchCV(lasso, {'alpha': alphas}, cv=5, scoring='neg_mean_squared_error')
lasso_grid.fit(X_train, y_train)

print(f"Lasso - Best alpha: {lasso_grid.best_params_['alpha']}")

# Test set evaluation
ridge_pred = ridge_grid.predict(X_test)
lasso_pred = lasso_grid.predict(X_test)

ridge_mse = mean_squared_error(y_test, ridge_pred)
lasso_mse = mean_squared_error(y_test, lasso_pred)

print(f"\nTest MSE - Ridge: {ridge_mse:.6f}")
print(f"Test MSE - Lasso: {lasso_mse:.6f}")


NameError: name 'X_train' is not defined

## Task 2, Step 3: Regularization Experiments (L1 vs L2)

In [11]:

# Train models with best parameters
ridge_best = Ridge(alpha=ridge_grid.best_params_['alpha'])
lasso_best = Lasso(alpha=lasso_grid.best_params_['alpha'], max_iter=10000)

ridge_best.fit(X_train, y_train)
lasso_best.fit(X_train, y_train)

# Compare coefficients
print("\nCoefficient Comparison (first 10 features):")
print("Feature\tBaseline\t\tRidge\t\t\tLasso")
for i in range(10):
    print(f"{i}\t{model.coef_[i]:.6f}\t{ridge_best.coef_[i]:.6f}\t{lasso_best.coef_[i]:.6f}")

# Count zero coefficients
zero_lasso = sum(lasso_best.coef_ == 0)
print(f"\nZero coefficients in Lasso: {zero_lasso}/{len(lasso_best.coef_)}")

# Performance comparison
ridge_train_mse = mean_squared_error(y_train, ridge_best.predict(X_train))
ridge_test_mse = mean_squared_error(y_test, ridge_best.predict(X_test))

lasso_train_mse = mean_squared_error(y_train, lasso_best.predict(X_train))
lasso_test_mse = mean_squared_error(y_test, lasso_best.predict(X_test))

print(f"\nPerformance Comparison:")
print(f"{'Model':<10} {'Train MSE':<15} {'Test MSE':<15}")
print("-" * 40)
print(f"{'Baseline':<10} {mse_train:<15.6f} {mse_test:<15.6f}")
print(f"{'Ridge':<10} {ridge_train_mse:<15.6f} {ridge_test_mse:<15.6f}")
print(f"{'Lasso':<10} {lasso_train_mse:<15.6f} {lasso_test_mse:<15.6f}")

print("\nDiscussion:")
print("1. L1 produces sparse coefficients (feature selection)")
print("2. L2 shrinks coefficients without zeroing them")
print("3. Regularization reduces variance, prevents overfitting")
print("4. Excessive regularization increases bias")



AttributeError: 'GridSearchCV' object has no attribute 'best_params_'

# PART 2: CLASSIFICATION TASK (Breast Cancer)

## Task 1: Load and Split Dataset

In [12]:

from sklearn.datasets import load_breast_cancer

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")


Training set size: (455, 30)
Test set size: (114, 30)


## Task 2, Step 1: Baseline Model (No Regularization)

In [13]:


from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Build Logistic Regression model
logreg_baseline = LogisticRegression(penalty=None, max_iter=10000, solver='lbfgs')
logreg_baseline.fit(X_train, y_train)

# Observe coefficients
print("Number of coefficients:", len(logreg_baseline.coef_[0]))

# Compute accuracy
y_train_pred = logreg_baseline.predict(X_train)
y_test_pred = logreg_baseline.predict(X_test)

train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print(f"\nTraining Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print("Step 1 completed ✓")

C:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Number of coefficients: 30

Training Accuracy: 0.9912
Test Accuracy: 0.9825
Step 1 completed ✓


C:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 10000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=10000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## Task 2, Step 2: Hyperparameter Tuning

In [14]:
from sklearn.model_selection import GridSearchCV

# Define parameter grid
param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100, 1000],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear']
}

# Grid Search
logreg = LogisticRegression(max_iter=10000)
grid_search = GridSearchCV(logreg, param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best CV accuracy:", grid_search.best_score_)

# Evaluate on test set
best_model = grid_search.best_estimator_
test_acc_tuned = accuracy_score(y_test, best_model.predict(X_test))
print(f"Test Accuracy with best model: {test_acc_tuned:.4f}")
print("Step 2 completed ✓")

C:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
C:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penal

Best parameters: {'C': 100, 'penalty': 'l1', 'solver': 'liblinear'}
Best CV accuracy: 0.9670329670329672
Test Accuracy with best model: 0.9825
Step 2 completed ✓


## Task 2, Step 3: Regularization Experiments (L1 vs L2)

In [16]:

# Get best parameters for L1 and L2
cv_results = grid_search.cv_results_
best_l1_idx = None
best_l2_idx = None

for i, params in enumerate(cv_results['params']):
    if params['penalty'] == 'l1':
        if best_l1_idx is None or cv_results['mean_test_score'][i] > cv_results['mean_test_score'][best_l1_idx]:
            best_l1_idx = i
    elif params['penalty'] == 'l2':
        if best_l2_idx is None or cv_results['mean_test_score'][i] > cv_results['mean_test_score'][best_l2_idx]:
            best_l2_idx = i

best_l1_params = cv_results['params'][best_l1_idx]
best_l2_params = cv_results['params'][best_l2_idx]

print("Best L1 parameters:", best_l1_params)
print("Best L2 parameters:", best_l2_params)

# Train models with best parameters
logreg_l1 = LogisticRegression(**best_l1_params, max_iter=10000)
logreg_l2 = LogisticRegression(**best_l2_params, max_iter=10000)

logreg_l1.fit(X_train, y_train)
logreg_l2.fit(X_train, y_train)

# Compare coefficients
print("\nCoefficient Comparison (first 10 features):")
print("Feature\t\tBaseline\t\tL1\t\t\tL2")
for i in range(10):
    print(f"{i}\t\t{logreg_baseline.coef_[0][i]:.6f}\t\t{logreg_l1.coef_[0][i]:.6f}\t\t{logreg_l2.coef_[0][i]:.6f}")

# Count zero coefficients
zero_l1 = sum(logreg_l1.coef_[0] == 0)
print(f"\nZero coefficients in L1: {zero_l1}/{len(logreg_l1.coef_[0])}")

# Evaluate and compare accuracy
acc_l1_train = accuracy_score(y_train, logreg_l1.predict(X_train))
acc_l1_test = accuracy_score(y_test, logreg_l1.predict(X_test))

acc_l2_train = accuracy_score(y_train, logreg_l2.predict(X_train))
acc_l2_test = accuracy_score(y_test, logreg_l2.predict(X_test))

print(f"\nAccuracy Comparison:")
print(f"{'Model':<15} {'Train Acc':<15} {'Test Acc':<15}")
print("-" * 45)
print(f"{'Baseline':<15} {train_acc:<15.4f} {test_acc:<15.4f}")
print(f"{'L1':<15} {acc_l1_train:<15.4f} {acc_l1_test:<15.4f}")
print(f"{'L2':<15} {acc_l2_train:<15.4f} {acc_l2_test:<15.4f}")

print("\nDiscussion:")
print("1. L1 produces sparse coefficients (feature selection)")
print("2. L2 shrinks all coefficients but rarely zero")
print("3. Regularization reduces variance and mitigates overfitting")
print("4. Overly strong regularization may increase bias")


C:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


Best L1 parameters: {'C': 100, 'penalty': 'l1', 'solver': 'liblinear'}
Best L2 parameters: {'C': 10, 'penalty': 'l2', 'solver': 'liblinear'}

Coefficient Comparison (first 10 features):
Feature		Baseline		L1			L2
0		3.451013		0.721878		4.398356
1		-0.055885		-0.108133		0.296195
2		0.147424		0.101079		-0.498847
3		-0.038344		-0.002227		-0.008948
4		-20.238911		0.000000		-0.546015
5		16.500126		47.250244		-0.819821
6		-27.414205		-12.085268		-1.604845
7		-47.873397		-135.973960		-1.233964
8		17.421976		19.785044		-0.739526
9		3.129449		0.000000		-0.022000

Zero coefficients in L1: 9/30

Accuracy Comparison:
Model           Train Acc       Test Acc       
---------------------------------------------
Baseline        0.9912          0.9825         
L1              0.9890          0.9825         
L2              0.9692          0.9561         

Discussion:
1. L1 produces sparse coefficients (feature selection)
2. L2 shrinks all coefficients but rarely zero
3. Regularization reduces variance

C:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
